# Metrics consolidation

Notebook for aggregating factuality, expert-writing, and SurGE evaluation runs from `results/scores/`. It mirrors the main plots from `app/main.py` and always keeps the corresponding numerical tables next to the figures.

Default behavior: load the latest run for each `(dataset, model, metric family)`. Set `LOAD_ALL_RUNS = True` to inspect all run variants.

In [2]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SCORES_DIR = ROOT / "results" / "scores"
GENERATIONS_DIR = ROOT / "results" / "generations"

DATASET = "SurGE"
KINDS = ("factuality", "expert", "surge")

# Switches for analysis variants.
LOAD_ALL_RUNS = False
PREFERRED_FACTUALITY_SUFFIX = None  # e.g. "_gemma-4-31b_run1_section_full_text_or_abstract_concat_topk5_abs05"
SELECTED_MODELS = None              # e.g. ["reference", "perplexity_dr", "surveyforge"]
PAIRWISE_MODELS = None              # e.g. ("reference", "perplexity_dr")
SURVEY_ID = None                    # e.g. "0"; None selects the first available survey

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)


In [3]:
EXPERT_SCALAR_METRICS = [
    ("m_crit", "Critical (m_crit)"),
    ("m_comp_total", "Comparative (m_comp_total)"),
    ("m_open", "Open questions (m_open)"),
    ("m_mod", "Modality entropy (m_mod)"),
]

EXPERT_RADAR_METRICS = [
    ("m_crit", "Critical"),
    ("m_comp_total", "Comparative"),
    ("m_open", "Open questions"),
    ("m_mod", "Modality entropy"),
]

FACTUALITY_SCALAR_METRICS = [
    ("cit_correct_overall", "Overall (cit_correct)"),
    ("cit_correct_A", "A - topical"),
    ("cit_correct_B", "B - methodological"),
    ("cit_correct_C", "C - quantitative"),
    ("cit_correct_D", "D - critical / comparative"),
]

FACTUALITY_RADAR_METRICS = [
    ("cit_correct_overall", "Overall"),
    ("cit_correct_A", "A"),
    ("cit_correct_B", "B"),
    ("cit_correct_C", "C"),
    ("cit_correct_D", "D"),
]

SURGE_SCALAR_METRICS = [
    ("rouge_1", "ROUGE-1"),
    ("rouge_2", "ROUGE-2"),
    ("rouge_l", "ROUGE-L"),
    ("bleu", "BLEU"),
    ("sh_recall", "Semantic Scholar recall"),
    ("relevance_paper", "Relevance: paper"),
    ("relevance_section", "Relevance: section"),
    ("relevance_sentence", "Relevance: sentence"),
    ("structure_quality", "Structure quality"),
    ("logic", "Logic"),
    ("citation_count", "Citation count"),
    ("corpus_match_rate", "Corpus match rate"),
    ("coverage", "Coverage"),
    ("reference_self_cited", "Reference self-cited"),
]

METRICS_BY_KIND = {
    "expert": EXPERT_SCALAR_METRICS,
    "factuality": FACTUALITY_SCALAR_METRICS,
    "surge": SURGE_SCALAR_METRICS,
}

RADAR_COLORS = [
    "#2196F3", "#E91E63", "#4CAF50", "#FF9800",
    "#9C27B0", "#00BCD4", "#FF5722", "#8BC34A",
]

MODALITY_COLORS = {
    "1": "#b10026",
    "2": "#e31a1c",
    "3": "#fd8d3c",
    "4": "#fecc5c",
    "5": "#2b8cbe",
}
MODALITY_LABELS = {
    "1": "1 - categorical",
    "2": "2",
    "3": "3 - hedged",
    "4": "4",
    "5": "5 - explicit uncertainty",
}

FACTUALITY_CATEGORY_COLORS = {
    "A": "#fecc5c",
    "B": "#41b6c4",
    "C": "#7fcdbb",
    "D": "#b10026",
}
FACTUALITY_CATEGORY_LABELS = {
    "A": "A - topical",
    "B": "B - methodological",
    "C": "C - quantitative",
    "D": "D - critical",
}


In [4]:
def numeric_stem_key(path: Path) -> tuple[int, int | str]:
    stem = path.stem
    return (0, int(stem)) if stem.isdigit() else (1, stem)


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def to_float(value) -> float | None:
    if isinstance(value, bool):
        return float(value)
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        if math.isnan(float(value)):
            return None
        return float(value)
    return None


def canonical_model_map(dataset: str) -> dict[str, str]:
    out: dict[str, str] = {}
    prefix = dataset + "_"
    if not GENERATIONS_DIR.exists():
        return out
    for path in GENERATIONS_DIR.iterdir():
        if path.is_dir() and path.name.startswith(prefix):
            model = path.name[len(prefix):]
            out.setdefault(model.lower(), model)
    return out


def canonical_model(dataset: str, raw_model: str) -> str:
    return canonical_model_map(dataset).get(raw_model.lower(), raw_model)


RUN_RE = re.compile(
    r"^(?P<dataset>[^_]+)_(?P<model>.+?)_(?P<kind>factuality|expert|surge)(?P<suffix>.*)$",
    re.IGNORECASE,
)


def discover_runs(kinds: tuple[str, ...] = KINDS) -> pd.DataFrame:
    rows = []
    if not SCORES_DIR.exists():
        return pd.DataFrame()
    for run_dir in SCORES_DIR.iterdir():
        if not run_dir.is_dir():
            continue
        match = RUN_RE.match(run_dir.name)
        if not match:
            continue
        kind = match.group("kind").lower()
        if kind not in kinds:
            continue
        dataset = match.group("dataset")
        raw_model = match.group("model")
        model = canonical_model(dataset, raw_model)
        rows.append({
            "run_name": run_dir.name,
            "path": str(run_dir),
            "dataset": dataset,
            "raw_model": raw_model,
            "model": model,
            "model_key": model.lower(),
            "kind": kind,
            "suffix": match.group("suffix") or "",
            "mtime": pd.Timestamp.fromtimestamp(run_dir.stat().st_mtime),
            "n_files": len([p for p in run_dir.glob("*.json") if p.stem != "summary"]),
            "has_summary_csv": (run_dir / "summary.csv").exists(),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values(["dataset", "model_key", "kind", "mtime"], ascending=[True, True, True, False]).reset_index(drop=True)


def select_runs(runs: pd.DataFrame) -> pd.DataFrame:
    if runs.empty:
        return runs
    df = runs[runs["dataset"].str.lower() == DATASET.lower()].copy()
    if SELECTED_MODELS:
        keys = {m.lower() for m in SELECTED_MODELS}
        df = df[df["model_key"].isin(keys)]
    if PREFERRED_FACTUALITY_SUFFIX:
        factuality = df["kind"] == "factuality"
        df = df[(~factuality) | (df["suffix"] == PREFERRED_FACTUALITY_SUFFIX)]
    if LOAD_ALL_RUNS:
        return df.reset_index(drop=True)
    df = df.sort_values("mtime", ascending=False)
    df = df.drop_duplicates(["dataset", "model_key", "kind"], keep="first")
    return df.sort_values(["model_key", "kind"]).reset_index(drop=True)


In [5]:
def iter_score_files(run_dir: Path):
    for path in sorted(run_dir.glob("*.json"), key=numeric_stem_key):
        if path.stem == "summary":
            continue
        yield path


def metric_values(kind: str, data: dict) -> dict[str, float]:
    if kind == "surge":
        raw = data.get("scores", {})
    else:
        keys = [key for key, _ in METRICS_BY_KIND[kind]]
        raw = {key: data.get(key) for key in keys}
        if "n_claims" in data:
            raw["n_claims"] = data.get("n_claims")
        if kind == "factuality" and "n_supported" in data:
            raw["n_supported"] = data.get("n_supported")
    out = {}
    for key, value in raw.items():
        num = to_float(value)
        if num is not None:
            out[key] = num
    return out


def load_metric_tables(run_df: pd.DataFrame):
    long_rows = []
    survey_rows = []
    factuality_category_rows = []
    expert_modality_rows = []

    for run in run_df.itertuples(index=False):
        run_dir = Path(run.path)
        for path in iter_score_files(run_dir):
            try:
                data = load_json(path)
            except Exception as exc:
                print(f"skip {path}: {exc}")
                continue

            survey_id = str(data.get("survey_id") or data.get("id") or path.stem)
            query = data.get("query") or data.get("survey_title") or ""
            metrics = metric_values(run.kind, data)
            base = {
                "run_name": run.run_name,
                "dataset": run.dataset,
                "raw_model": run.raw_model,
                "model": run.model,
                "model_key": run.model_key,
                "kind": run.kind,
                "suffix": run.suffix,
                "survey_id": survey_id,
                "query": query,
            }
            survey_rows.append({**base, **metrics})
            for metric, value in metrics.items():
                long_rows.append({**base, "metric": metric, "value": value})

            if run.kind == "factuality":
                counts = data.get("category_counts", {})
                total = to_float(data.get("n_claims")) or 0.0
                for category in ("A", "B", "C", "D"):
                    sub = counts.get(category) if isinstance(counts, dict) else None
                    if not isinstance(sub, dict):
                        continue
                    n = to_float(sub.get("n")) or 0.0
                    n_supported = to_float(sub.get("n_supported"))
                    factuality_category_rows.append({
                        **base,
                        "category": category,
                        "n": n,
                        "n_supported": n_supported,
                        "share": n / total if total else np.nan,
                        "support_rate": n_supported / n if n_supported is not None and n else np.nan,
                    })

            if run.kind == "expert":
                dist = data.get("modality_dist")
                if isinstance(dist, dict):
                    total = sum(float(dist.get(str(level), 0) or 0) for level in range(1, 6))
                    if total > 0:
                        expert_modality_rows.append({
                            **base,
                            **{str(level): float(dist.get(str(level), 0) or 0) / total for level in range(1, 6)},
                            **{f"n_{level}": float(dist.get(str(level), 0) or 0) for level in range(1, 6)},
                        })

    return (
        pd.DataFrame(long_rows),
        pd.DataFrame(survey_rows),
        pd.DataFrame(factuality_category_rows),
        pd.DataFrame(expert_modality_rows),
    )


runs_all = discover_runs()
display(runs_all)

selected_runs = select_runs(runs_all)
display(selected_runs)

metric_long, survey_wide, factuality_categories, expert_modality = load_metric_tables(selected_runs)
print(f"loaded metric rows: {len(metric_long):,}")
display(survey_wide.head())


,run_name,path,dataset,raw_model,model,model_key,kind,suffix,mtime,n_files,has_summary_csv
0,SurGE_perplexity_dr_expert_gemma-4-31b_run1,/Users/kellesett/study/thesis/results/scores/S...,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,2026-05-02 23:25:03.360577,40,True
1,SurGE_perplexity_dr_factuality_gemma-4-31b_run...,/Users/kellesett/study/thesis/results/scores/S...,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,factuality,_gemma-4-31b_run1_section_abstract_per_ref,2026-05-02 23:25:11.261795,40,True
2,SurGE_perplexity_dr_factuality_gemma-4-31b_run...,/Users/kellesett/study/thesis/results/scores/S...,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,factuality,_gemma-4-31b_run1_section_full_text_or_abstrac...,2026-05-02 15:13:38.114058,40,True
3,SurGE_perplexity_dr_factuality_gemma-4-31b_run...,/Users/kellesett/study/thesis/results/scores/S...,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,factuality,_gemma-4-31b_run1_section_full_text_or_abstrac...,2026-05-02 15:13:37.840398,0,False
4,SurGE_perplexity_dr_factuality_gemma-4-31b_run...,/Users/kellesett/study/thesis/results/scores/S...,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,factuality,_gemma-4-31b_run1_section_full_text_or_abstrac...,2026-05-02 15:13:37.240502,0,False
5,SurGE_perplexity_dr_surge_gemma-4-31b_run1,/Users/kellesett/study/thesis/results/scores/S...,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,surge,_gemma-4-31b_run1,2026-05-02 03:00:38.618352,40,True
6,SurGE_reference_expert_gemma-4-31b_run1,/Users/kellesett/study/thesis/results/scores/S...,SurGE,reference,reference,reference,expert,_gemma-4-31b_run1,2026-05-02 23:26:02.421380,36,True
7,SurGE_reference_factuality_gemma-4-31b_run1_se...,/Users/kellesett/study/thesis/results/scores/S...,SurGE,reference,reference,reference,factuality,_gemma-4-31b_run1_section_full_text_or_abstrac...,2026-05-02 23:25:31.647716,36,True
8,SurGE_reference_factuality_gemma-4-31b_run1_se...,/Users/kellesett/study/thesis/results/scores/S...,SurGE,reference,reference,reference,factuality,_gemma-4-31b_run1_section_abstract_per_ref,2026-05-02 15:13:39.802970,16,True
9,SurGE_reference_factuality_gemma-4-31b_run1_se...,/Users/kellesett/study/thesis/results/scores/S...,SurGE,reference,reference,reference,factuality,_gemma-4-31b_run1_section_full_text_or_abstrac...,2026-05-02 15:13:39.436888,1,True


,run_name,path,dataset,raw_model,model,model_key,kind,suffix,mtime,n_files,has_summary_csv
0,SurGE_perplexity_dr_expert_gemma-4-31b_run1,/Users/kellesett/study/thesis/results/scores/S...,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,2026-05-02 23:25:03.360577,40,True
1,SurGE_perplexity_dr_factuality_gemma-4-31b_run...,/Users/kellesett/study/thesis/results/scores/S...,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,factuality,_gemma-4-31b_run1_section_abstract_per_ref,2026-05-02 23:25:11.261795,40,True
2,SurGE_perplexity_dr_surge_gemma-4-31b_run1,/Users/kellesett/study/thesis/results/scores/S...,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,surge,_gemma-4-31b_run1,2026-05-02 03:00:38.618352,40,True
3,SurGE_reference_expert_gemma-4-31b_run1,/Users/kellesett/study/thesis/results/scores/S...,SurGE,reference,reference,reference,expert,_gemma-4-31b_run1,2026-05-02 23:26:02.421380,36,True
4,SurGE_reference_factuality_gemma-4-31b_run1_se...,/Users/kellesett/study/thesis/results/scores/S...,SurGE,reference,reference,reference,factuality,_gemma-4-31b_run1_section_full_text_or_abstrac...,2026-05-02 23:25:31.647716,36,True
5,SurGE_reference_surge_gemma-4-31b_run1,/Users/kellesett/study/thesis/results/scores/S...,SurGE,reference,reference,reference,surge,_gemma-4-31b_run1,2026-05-02 17:16:52.928508,36,True
6,SurGE_SurveyForge_expert_gemma-4-31b_run1,/Users/kellesett/study/thesis/results/scores/S...,SurGE,SurveyForge,surveyforge,surveyforge,expert,_gemma-4-31b_run1,2026-05-02 23:26:01.230296,41,True
7,SurGE_surveyforge_factuality_gemma-4-31b_run1_...,/Users/kellesett/study/thesis/results/scores/S...,SurGE,surveyforge,surveyforge,surveyforge,factuality,_gemma-4-31b_run1_section_abstract_per_ref,2026-05-02 23:24:50.586913,38,True
8,SurGE_surveyforge_surge_gemma-4-31b_run1,/Users/kellesett/study/thesis/results/scores/S...,SurGE,surveyforge,surveyforge,surveyforge,surge,_gemma-4-31b_run1,2026-05-02 17:40:39.909428,40,True


loaded metric rows: 2,994


,run_name,dataset,raw_model,model,model_key,kind,suffix,survey_id,query,m_crit,m_comp_total,m_open,m_mod,n_claims,cit_correct_overall,cit_correct_A,cit_correct_B,cit_correct_C,cit_correct_D,n_supported,rouge_1,rouge_2,rouge_l,bleu,sh_recall,relevance_paper,relevance_section,relevance_sentence,structure_quality,logic,reference_self_cited,coverage,citation_count,corpus_match_rate
0,SurGE_perplexity_dr_expert_gemma-4-31b_run1,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,0,A Survey on Edge Computing Systems and Tools,0.1175,0.0989,0.0014,0.4890,698.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SurGE_perplexity_dr_expert_gemma-4-31b_run1,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,1,A Survey on Graph-Based Deep Learning \\ for C...,0.1103,0.0948,0.0121,0.4855,580.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SurGE_perplexity_dr_expert_gemma-4-31b_run1,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,2,A Survey of Moving Target Defenses for\\Networ...,0.1828,0.0448,0.0086,0.5420,580.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SurGE_perplexity_dr_expert_gemma-4-31b_run1,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,3,Explanation-Based Human Debugging of NLP Model...,0.1975,0.1215,0.0152,0.6506,395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SurGE_perplexity_dr_expert_gemma-4-31b_run1,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,4,Deep Learning for Image Super-resolution:\\A S...,0.1595,0.1496,0.0071,0.2490,702.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
def metric_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return (
        df.groupby(["dataset", "model", "kind", "metric"], dropna=False)["value"]
          .agg(n="count", mean="mean", std="std", min="min", q25=lambda s: s.quantile(0.25), median="median", q75=lambda s: s.quantile(0.75), max="max")
          .reset_index()
          .sort_values(["kind", "metric", "model"])
    )


summary = metric_summary(metric_long)
display(summary)

if not summary.empty:
    mean_matrix = summary.pivot_table(index=["kind", "metric"], columns="model", values="mean")
    display(mean_matrix)

if not survey_wide.empty:
    display(survey_wide.sort_values(["kind", "model", "survey_id"]))


,dataset,model,kind,metric,n,mean,std,min,q25,median,q75,max
0,SurGE,perplexity_dr,expert,m_comp_total,40,0.120853,0.036135,0.044800,0.098400,0.121500,0.146450,0.195500
26,SurGE,reference,expert,m_comp_total,36,0.052339,0.020510,0.014800,0.037225,0.052350,0.060825,0.102800
52,SurGE,surveyforge,expert,m_comp_total,41,0.094727,0.020953,0.041600,0.081900,0.094700,0.107000,0.136700
1,SurGE,perplexity_dr,expert,m_crit,40,0.179740,0.076925,0.079300,0.123100,0.155300,0.217975,0.422000
27,SurGE,reference,expert,m_crit,36,0.092778,0.048076,0.027300,0.064225,0.079700,0.097950,0.225200
53,SurGE,surveyforge,expert,m_crit,41,0.228415,0.072832,0.097800,0.177800,0.210100,0.253000,0.419800
2,SurGE,perplexity_dr,expert,m_mod,40,0.467553,0.104577,0.249000,0.398825,0.467500,0.537700,0.714500
28,SurGE,reference,expert,m_mod,36,0.465806,0.124280,0.254600,0.393900,0.457600,0.528925,0.869400
54,SurGE,surveyforge,expert,m_mod,41,0.609634,0.083431,0.407500,0.553200,0.602900,0.650000,0.802600
3,SurGE,perplexity_dr,expert,m_open,40,0.009150,0.006537,0.000000,0.004125,0.008550,0.012825,0.027000


model                            perplexity_dr    reference  surveyforge
kind       metric                                                       
expert     m_comp_total               0.120853     0.052339     0.094727
           m_crit                     0.179740     0.092778     0.228415
           m_mod                      0.467553     0.465806     0.609634
           m_open                     0.009150     0.004203     0.005522
           n_claims                 574.950000  1085.972222   867.902439
factuality cit_correct_A              0.421390     0.649539     0.377721
           cit_correct_B              0.316765     0.568119     0.291571
           cit_correct_C              0.434829     0.388558     0.700833
           cit_correct_D              0.377585     0.501881     0.296111
           cit_correct_overall        0.401597     0.592642     0.346666
           n_claims                 574.950000  1088.500000   870.921053
           n_supported              229.725000   639.083333   299.710526
surge      bleu                       6.657160    78.880837     9.474863
           citation_count            49.325000   177.833333    98.575000
           corpus_match_rate          0.505197     0.435892     1.000000
           coverage                   0.035948     0.980375     0.000000
           logic                      4.653659     4.002947     4.644687
           reference_self_cited       0.775000     0.000000     0.000000
           relevance_paper            0.250389     0.377818     0.500000
           relevance_section          0.257724     0.213246     0.383539
           relevance_sentence         0.262592     0.217102     0.439206
           rouge_1                    0.313394     0.886009     0.375541
           rouge_2                    0.049880     0.843885     0.058024
           rouge_l                    0.138397     0.874174     0.152056
           sh_recall                  0.929836     1.000637     0.962952
           structure_quality          1.775000     4.638889     1.925000

,run_name,dataset,raw_model,model,model_key,kind,suffix,survey_id,query,m_crit,m_comp_total,m_open,m_mod,n_claims,cit_correct_overall,cit_correct_A,cit_correct_B,cit_correct_C,cit_correct_D,n_supported,rouge_1,rouge_2,rouge_l,bleu,sh_recall,relevance_paper,relevance_section,relevance_sentence,structure_quality,logic,reference_self_cited,coverage,citation_count,corpus_match_rate
0,SurGE_perplexity_dr_expert_gemma-4-31b_run1,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,0,A Survey on Edge Computing Systems and Tools,0.1175,0.0989,0.0014,0.4890,698.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SurGE_perplexity_dr_expert_gemma-4-31b_run1,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,1,A Survey on Graph-Based Deep Learning \\ for C...,0.1103,0.0948,0.0121,0.4855,580.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,SurGE_perplexity_dr_expert_gemma-4-31b_run1,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,10,Recent Advances in Deep Learning Based Dialogu...,0.1676,0.1873,0.0169,0.4408,710.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,SurGE_perplexity_dr_expert_gemma-4-31b_run1,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,11,End-to-End Constrained Optimization Learning: ...,0.1300,0.0969,0.0198,0.5683,454.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,SurGE_perplexity_dr_expert_gemma-4-31b_run1,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,12,A Survey of Label-noise Representation Learnin...,0.1584,0.1215,0.0087,0.5603,461.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
312,SurGE_surveyforge_surge_gemma-4-31b_run1,SurGE,surveyforge,surveyforge,surveyforge,surge,_gemma-4-31b_run1,5,A survey of active learning algorithms for sup...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.358931,0.055233,0.147601,8.317579,0.965710,0.5,0.320652,0.461957,1.0,4.40,0.0,0.0,93.0,1.0
313,SurGE_surveyforge_surge_gemma-4-31b_run1,SurGE,surveyforge,surveyforge,surveyforge,surge,_gemma-4-31b_run1,6,Deep Learning for Deepfakes Creation and Detec...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.340565,0.043830,0.137393,5.103419,0.991192,0.5,0.384365,0.488599,2.0,4.75,0.0,0.0,117.0,1.0
314,SurGE_surveyforge_surge_gemma-4-31b_run1,SurGE,surveyforge,surveyforge,surveyforge,surge,_gemma-4-31b_run1,7,A Survey of Active Learning for Text Classific...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.362911,0.055429,0.149186,6.444509,0.984007,0.5,0.389362,0.487234,2.0,4.45,0.0,0.0,98.0,1.0
315,SurGE_surveyforge_surge_gemma-4-31b_run1,SurGE,surveyforge,surveyforge,surveyforge,surge,_gemma-4-31b_run1,8,Survey on Evaluation Methods for Dialogue Systems,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.361636,0.046566,0.148645,5.932812,0.965959,0.5,0.428839,0.447566,1.0,4.60,0.0,0.0,94.0,1.0


In [7]:
def specs_available(kind: str, df: pd.DataFrame) -> list[tuple[str, str]]:
    available = set(df["metric"].unique()) if not df.empty else set()
    return [(key, label) for key, label in METRICS_BY_KIND[kind] if key in available]


def make_box_strip(df: pd.DataFrame, metric: str, title: str, y_range: list[float] | None = None) -> go.Figure:
    fig = go.Figure()
    if df.empty:
        return fig
    for model in sorted(df["model"].unique(), key=str.lower):
        sub = df[(df["model"] == model) & (df["metric"] == metric)]
        if sub.empty:
            continue
        fig.add_trace(go.Box(
            y=sub["value"],
            name=model,
            boxpoints="all",
            jitter=0.35,
            pointpos=0,
            marker=dict(size=4, opacity=0.65),
            line=dict(width=1),
            customdata=sub["survey_id"],
            hovertemplate="<b>%{fullData.name}</b><br>value: %{y:.4f}<br>survey: %{customdata}<extra></extra>",
        ))
    fig.update_layout(
        title=title,
        yaxis_title="value",
        yaxis=dict(range=y_range) if y_range else None,
        showlegend=False,
        height=320,
        margin=dict(l=40, r=20, t=40, b=40),
    )
    return fig


def show_box_grid(df: pd.DataFrame, kind: str, y_range: list[float] | None = None) -> None:
    specs = specs_available(kind, df)
    for metric, label in specs:
        fig = make_box_strip(df, metric, label, y_range=y_range)
        if fig.data:
            fig.show()


def radar_chart(
    title: str,
    labels: list[str],
    series: list[tuple[str, list[float | None], str]],
    height: int = 430,
    axis_maxima: list[float] | None = None,
    show_axis_maxima: bool = False,
) -> go.Figure:
    if axis_maxima is not None and len(axis_maxima) != len(labels):
        axis_maxima = None
    axis_labels = labels
    if axis_maxima is not None and show_axis_maxima:
        axis_labels = [f"{label}<br>max {max_v:.3g}" for label, max_v in zip(labels, axis_maxima)]
    theta = axis_labels + [axis_labels[0]]
    fig = go.Figure()
    for name, values, color in series:
        if not any(v is not None and not pd.isna(v) for v in values):
            continue
        if axis_maxima is None:
            closed = [float(v) if v is not None and not pd.isna(v) else None for v in values + [values[0]]]
            customdata = None
            hovertemplate = "<b>%{fullData.name}</b><br>%{theta}: %{r:.3f}<extra></extra>"
        else:
            scaled = []
            raw = []
            for value, max_value in zip(values, axis_maxima):
                raw_value = float(value) if value is not None and not pd.isna(value) else None
                axis_max = float(max_value) if max_value and max_value > 0 else 1.0
                scaled.append(raw_value / axis_max if raw_value is not None else None)
                raw.append([raw_value, axis_max])
            closed = scaled + [scaled[0]]
            customdata = raw + [raw[0]]
            hovertemplate = "<b>%{fullData.name}</b><br>%{theta}<br>value: %{customdata[0]:.3f}<br>axis max: %{customdata[1]:.3f}<br>scaled: %{r:.3f}<extra></extra>"
        fig.add_trace(go.Scatterpolar(
            r=closed,
            theta=theta,
            name=name,
            fill="toself",
            opacity=0.72,
            line=dict(color=color, width=2),
            marker=dict(size=5, color=color),
            connectgaps=True,
            customdata=customdata,
            hovertemplate=hovertemplate,
        ))
    fig.update_layout(
        title=title,
        height=height,
        margin=dict(l=30, r=30, t=50, b=30),
        polar=dict(radialaxis=dict(visible=True, range=[0, 1], tickformat=".1f", gridcolor="rgba(150,150,150,0.25)"), angularaxis=dict(gridcolor="rgba(150,150,150,0.25)")),
        legend=dict(orientation="h", y=-0.12),
    )
    return fig


def mean_values(df: pd.DataFrame, metric_specs: list[tuple[str, str]]) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    table = (
        df[df["metric"].isin([key for key, _ in metric_specs])]
        .groupby(["model", "metric"])["value"]
        .mean()
        .unstack("metric")
    )
    return table.reindex(columns=[key for key, _ in metric_specs if key in table.columns])


## SurGE metrics

SurGE scores are shown as per-metric distributions across surveys. The table reports the same values aggregated by model.

In [8]:
surge_df = metric_long[metric_long["kind"] == "surge"].copy() if not metric_long.empty else pd.DataFrame()
surge_summary = metric_summary(surge_df)
display(surge_summary)

if not surge_df.empty:
    display(mean_values(surge_df, SURGE_SCALAR_METRICS))
    show_box_grid(surge_df, "surge")
else:
    print("No SurGE metrics loaded.")


,dataset,model,kind,metric,n,mean,std,min,q25,median,q75,max
0,SurGE,perplexity_dr,surge,bleu,40,6.657160,2.240219,3.651419,4.808229,6.495251,7.570009,13.483344
14,SurGE,reference,surge,bleu,36,78.880837,11.577119,30.716544,76.331715,82.140450,85.004575,93.449415
28,SurGE,surveyforge,surge,bleu,40,9.474863,3.175529,5.103419,6.461443,9.278992,11.371838,16.448668
1,SurGE,perplexity_dr,surge,citation_count,40,49.325000,1.474353,44.000000,49.000000,50.000000,50.000000,50.000000
15,SurGE,reference,surge,citation_count,36,177.833333,98.738182,49.000000,101.000000,157.000000,235.250000,467.000000
29,SurGE,surveyforge,surge,citation_count,40,98.575000,13.733522,65.000000,93.000000,97.500000,109.000000,133.000000
2,SurGE,perplexity_dr,surge,corpus_match_rate,40,0.505197,0.188275,0.140000,0.336625,0.515100,0.650000,0.820000
16,SurGE,reference,surge,corpus_match_rate,36,0.435892,0.229478,0.035700,0.300625,0.450200,0.591700,0.837700
30,SurGE,surveyforge,surge,corpus_match_rate,40,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3,SurGE,perplexity_dr,surge,coverage,40,0.035948,0.039533,0.000000,0.000000,0.029950,0.048200,0.166700


metric,rouge_1,rouge_2,rouge_l,bleu,sh_recall,relevance_paper,relevance_section,relevance_sentence,structure_quality,logic,citation_count,corpus_match_rate,coverage,reference_self_cited
model,,,,,,,,,,,,,,
perplexity_dr,0.313394,0.049880,0.138397,6.657160,0.929836,0.250389,0.257724,0.262592,1.775000,4.653659,49.325000,0.505197,0.035948,0.775
reference,0.886009,0.843885,0.874174,78.880837,1.000637,0.377818,0.213246,0.217102,4.638889,4.002947,177.833333,0.435892,0.980375,0.000
surveyforge,0.375541,0.058024,0.152056,9.474863,0.962952,0.500000,0.383539,0.439206,1.925000,4.644687,98.575000,1.000000,0.000000,0.000


## Expert-writing metrics

This reproduces the viewer's scalar boxplots and the modality distribution plots. `m_comp_valid` is intentionally absent because the current metric implementation does not include it.

In [11]:
expert_df = metric_long[metric_long["kind"] == "expert"].copy() if not metric_long.empty else pd.DataFrame()
expert_summary = metric_summary(expert_df)
display(expert_summary)

if not expert_df.empty:
    display(mean_values(expert_df, EXPERT_SCALAR_METRICS))
    show_box_grid(expert_df, "expert")
else:
    print("No expert metrics loaded.")

if not expert_modality.empty:
    modality_means = (
        expert_modality.groupby("model")[["1", "2", "3", "4", "5"]]
        .mean()
        .sort_index()
    )
    display(modality_means)

    modality_series = []
    for i, model in enumerate(modality_means.index):
        modality_series.append((model, [float(modality_means.loc[model, str(level)]) for level in range(1, 6)], RADAR_COLORS[i % len(RADAR_COLORS)]))
    modality_axis_maxima = []
    for idx in range(5):
        axis_values = [values[idx] for _, values, _ in modality_series if values[idx] is not None and not pd.isna(values[idx])]
        axis_max = max(axis_values) * 1.1 if axis_values else 1.0
        modality_axis_maxima.append(axis_max if axis_max > 0 else 1.0)
    radar_chart(
        "Average modality mix",
        [MODALITY_LABELS[str(level)] for level in range(1, 6)],
        modality_series,
        axis_maxima=modality_axis_maxima,
        show_axis_maxima=True,
    ).show()

    fig = go.Figure()
    for level in ("1", "2", "3", "4", "5"):
        fig.add_trace(go.Bar(
            x=modality_means.index,
            y=modality_means[level],
            name=MODALITY_LABELS[level],
            marker_color=MODALITY_COLORS[level],
            hovertemplate=f"%{{x}}<br>{MODALITY_LABELS[level]}: %{{y:.3f}}<extra></extra>",
        ))
    fig.update_layout(barmode="stack", yaxis_title="mean share", yaxis=dict(range=[0, 1.02]), legend_title="modality", height=400, margin=dict(l=40, r=20, t=20, b=40))
    fig.show()
else:
    print("No modality distributions loaded.")


,dataset,model,kind,metric,n,mean,std,min,q25,median,q75,max
0,SurGE,perplexity_dr,expert,m_comp_total,40,0.120853,0.036135,0.0448,0.098400,0.12150,0.146450,0.1955
5,SurGE,reference,expert,m_comp_total,36,0.052339,0.020510,0.0148,0.037225,0.05235,0.060825,0.1028
10,SurGE,surveyforge,expert,m_comp_total,41,0.094727,0.020953,0.0416,0.081900,0.09470,0.107000,0.1367
1,SurGE,perplexity_dr,expert,m_crit,40,0.179740,0.076925,0.0793,0.123100,0.15530,0.217975,0.4220
6,SurGE,reference,expert,m_crit,36,0.092778,0.048076,0.0273,0.064225,0.07970,0.097950,0.2252
11,SurGE,surveyforge,expert,m_crit,41,0.228415,0.072832,0.0978,0.177800,0.21010,0.253000,0.4198
2,SurGE,perplexity_dr,expert,m_mod,40,0.467553,0.104577,0.2490,0.398825,0.46750,0.537700,0.7145
7,SurGE,reference,expert,m_mod,36,0.465806,0.124280,0.2546,0.393900,0.45760,0.528925,0.8694
12,SurGE,surveyforge,expert,m_mod,41,0.609634,0.083431,0.4075,0.553200,0.60290,0.650000,0.8026
3,SurGE,perplexity_dr,expert,m_open,40,0.009150,0.006537,0.0000,0.004125,0.00855,0.012825,0.0270


metric,m_crit,m_comp_total,m_open,m_mod
model,,,,
perplexity_dr,0.179740,0.120853,0.009150,0.467553
reference,0.092778,0.052339,0.004203,0.465806
surveyforge,0.228415,0.094727,0.005522,0.609634


,1,2,3,4,5
model,,,,,
perplexity_dr,0.865221,0.014635,0.108720,0.010757,0.000667
reference,0.871761,0.024805,0.092523,0.010730,0.000181
surveyforge,0.793380,0.015502,0.175942,0.014776,0.000400


## Factuality metrics

This reproduces the viewer's CitCorrect scalar distributions, claim-category radar, and category stacked bar. Support rates are kept as a table for inspection.

In [12]:
factuality_df = metric_long[metric_long["kind"] == "factuality"].copy() if not metric_long.empty else pd.DataFrame()
factuality_summary = metric_summary(factuality_df)
display(factuality_summary)

if not factuality_df.empty:
    display(mean_values(factuality_df, FACTUALITY_SCALAR_METRICS))
    show_box_grid(factuality_df, "factuality", y_range=[0, 1.02])
else:
    print("No factuality metrics loaded.")

if not factuality_categories.empty:
    category_share = (
        factuality_categories.groupby(["model", "category"])["share"]
        .mean()
        .unstack("category")
        .reindex(columns=["A", "B", "C", "D"])
    )
    support_rate = (
        factuality_categories.groupby(["model", "category"])["support_rate"]
        .mean()
        .unstack("category")
        .reindex(columns=["A", "B", "C", "D"])
    )
    display(category_share)
    display(support_rate)

    category_series = []
    for i, model in enumerate(category_share.index):
        values = [float(category_share.loc[model, cat]) if not pd.isna(category_share.loc[model, cat]) else None for cat in ("A", "B", "C", "D")]
        category_series.append((model, values, RADAR_COLORS[i % len(RADAR_COLORS)]))
    radar_chart("Average claim-category mix", [FACTUALITY_CATEGORY_LABELS[cat] for cat in ("A", "B", "C", "D")], category_series).show()

    fig = go.Figure()
    for cat in ("A", "B", "C", "D"):
        fig.add_trace(go.Bar(
            x=category_share.index,
            y=category_share[cat],
            name=FACTUALITY_CATEGORY_LABELS[cat],
            marker_color=FACTUALITY_CATEGORY_COLORS[cat],
            hovertemplate=f"%{{x}}<br>{FACTUALITY_CATEGORY_LABELS[cat]}: %{{y:.3f}}<extra></extra>",
        ))
    fig.update_layout(barmode="stack", yaxis_title="mean share", yaxis=dict(range=[0, 1.02]), legend_title="category", height=400, margin=dict(l=40, r=20, t=20, b=40))
    fig.show()
else:
    print("No factuality category counts loaded.")


,dataset,model,kind,metric,n,mean,std,min,q25,median,q75,max
0,SurGE,perplexity_dr,factuality,cit_correct_A,40,0.421390,0.080059,0.2597,0.384325,0.42075,0.474250,0.6713
7,SurGE,reference,factuality,cit_correct_A,36,0.649539,0.146299,0.2764,0.607375,0.64545,0.755950,0.8603
14,SurGE,surveyforge,factuality,cit_correct_A,38,0.377721,0.058220,0.2570,0.327500,0.37765,0.412125,0.5161
1,SurGE,perplexity_dr,factuality,cit_correct_B,40,0.316765,0.158041,0.0370,0.206725,0.29465,0.415475,0.7640
8,SurGE,reference,factuality,cit_correct_B,36,0.568119,0.165785,0.1426,0.479900,0.55910,0.682425,0.8591
15,SurGE,surveyforge,factuality,cit_correct_B,38,0.291571,0.084426,0.1656,0.240125,0.28010,0.329350,0.5000
2,SurGE,perplexity_dr,factuality,cit_correct_C,38,0.434829,0.290135,0.0000,0.200000,0.42670,0.660750,1.0000
9,SurGE,reference,factuality,cit_correct_C,33,0.388558,0.225421,0.0000,0.241400,0.36570,0.555600,0.9286
16,SurGE,surveyforge,factuality,cit_correct_C,30,0.700833,0.367415,0.0000,0.541675,0.80000,1.000000,1.0000
3,SurGE,perplexity_dr,factuality,cit_correct_D,40,0.377585,0.103579,0.1696,0.307950,0.37575,0.459575,0.5750


metric,cit_correct_overall,cit_correct_A,cit_correct_B,cit_correct_C,cit_correct_D
model,,,,,
perplexity_dr,0.401597,0.421390,0.316765,0.434829,0.377585
reference,0.592642,0.649539,0.568119,0.388558,0.501881
surveyforge,0.346666,0.377721,0.291571,0.700833,0.296111


category,A,B,C,D
model,,,,
perplexity_dr,0.688286,0.136359,0.023443,0.151912
reference,0.490370,0.341994,0.042758,0.124878
surveyforge,0.625949,0.177388,0.003723,0.192940


category,A,B,C,D
model,,,,
perplexity_dr,0.421387,0.316757,0.434825,0.377584
reference,0.649535,0.568116,0.388555,0.501873
surveyforge,0.377721,0.291571,0.700833,0.296108


## Full-text-covered factuality profile

Viewer-side recomputation: keep only claims whose citation scope is fully covered by cached full text. If the corresponding source files are absent, this section stays empty.

In [ ]:
def load_full_text_ref_ids(dataset: str, raw_model: str, survey_id: str) -> set[int] | None:
    path = GENERATIONS_DIR / f"{dataset}_{raw_model}" / "sources" / f"{survey_id}_sources.json"
    if not path.exists():
        return None
    try:
        data = load_json(path)
    except Exception:
        return None
    refs = data.get("refs") if isinstance(data, dict) else None
    if isinstance(refs, dict):
        refs_iter = refs.values()
    elif isinstance(refs, list):
        refs_iter = refs
    else:
        return None
    out = set()
    for ref in refs_iter:
        if not isinstance(ref, dict):
            continue
        try:
            idx = int(ref.get("idx"))
        except (TypeError, ValueError):
            continue
        text = ref.get("text") or ref.get("full_text")
        if isinstance(text, str) and text.strip():
            out.add(idx)
    return out


def claim_scope_refs(claim: dict) -> list[int]:
    refs = claim.get("scope_citations")
    if not isinstance(refs, list):
        return []
    out = []
    for ref in refs:
        try:
            out.append(int(ref))
        except (TypeError, ValueError):
            pass
    return out


def cit_correct_profile_from_claims(claims: list[dict]) -> dict[str, float | None]:
    def rate(subset: list[dict]) -> float | None:
        if not subset:
            return None
        return sum(1 for claim in subset if claim.get("supported") is True) / len(subset)
    profile = {"cit_correct_overall": rate(claims)}
    for cat in ("A", "B", "C", "D"):
        profile[f"cit_correct_{cat}"] = rate([claim for claim in claims if claim.get("category") == cat])
    return profile


def build_full_text_profile(run_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    fact_runs = run_df[run_df["kind"] == "factuality"] if not run_df.empty else pd.DataFrame()
    for run in fact_runs.itertuples(index=False):
        for path in iter_score_files(Path(run.path)):
            try:
                data = load_json(path)
            except Exception:
                continue
            survey_id = str(data.get("survey_id") or data.get("id") or path.stem)
            full_text_ids = load_full_text_ref_ids(run.dataset, run.raw_model, survey_id)
            claims = data.get("claims")
            if not full_text_ids or not isinstance(claims, list):
                continue
            kept = [claim for claim in claims if isinstance(claim, dict) and claim_scope_refs(claim) and all(ref in full_text_ids for ref in claim_scope_refs(claim))]
            profile = cit_correct_profile_from_claims(kept)
            for metric, value in profile.items():
                if value is not None:
                    rows.append({
                        "dataset": run.dataset,
                        "raw_model": run.raw_model,
                        "model": run.model,
                        "run_name": run.run_name,
                        "survey_id": survey_id,
                        "metric": metric,
                        "value": float(value),
                        "n_claims": len(kept),
                    })
    return pd.DataFrame(rows)


full_text_profile = build_full_text_profile(selected_runs)
display(metric_summary(full_text_profile.assign(kind="factuality_full_text")) if not full_text_profile.empty else full_text_profile)

if not full_text_profile.empty:
    table = mean_values(full_text_profile.assign(kind="factuality"), FACTUALITY_RADAR_METRICS)
    display(table)
    series = []
    for i, model in enumerate(table.index):
        values = [float(table.loc[model, key]) if key in table.columns and not pd.isna(table.loc[model, key]) else None for key, _ in FACTUALITY_RADAR_METRICS]
        series.append((model, values, RADAR_COLORS[i % len(RADAR_COLORS)]))
    radar_chart("CitCorrect profile: full-text-covered scopes only", [label for _, label in FACTUALITY_RADAR_METRICS], series).show()
else:
    print("No full-text-covered factuality profile available.")


,dataset,model,kind,metric,n,mean,std,min,q25,median,q75,max
0,SurGE,perplexity_dr,factuality_full_text,cit_correct_A,39,0.413237,0.080341,0.190476,0.380055,0.423841,0.459856,0.608333
5,SurGE,reference,factuality_full_text,cit_correct_A,32,0.611337,0.246655,0.000000,0.555556,0.698875,0.750000,1.000000
10,SurGE,surveyforge,factuality_full_text,cit_correct_A,38,0.352330,0.077863,0.147493,0.322222,0.377931,0.403738,0.487985
1,SurGE,perplexity_dr,factuality_full_text,cit_correct_B,39,0.322260,0.162558,0.043478,0.206395,0.295455,0.419841,0.764045
6,SurGE,reference,factuality_full_text,cit_correct_B,31,0.606276,0.233994,0.000000,0.500000,0.615385,0.741279,1.000000
11,SurGE,surveyforge,factuality_full_text,cit_correct_B,38,0.291343,0.086151,0.152174,0.240140,0.281005,0.336890,0.500000
2,SurGE,perplexity_dr,factuality_full_text,cit_correct_C,37,0.466654,0.290538,0.000000,0.200000,0.434783,0.666667,1.000000
7,SurGE,reference,factuality_full_text,cit_correct_C,20,0.507057,0.359969,0.000000,0.250000,0.500000,0.875000,1.000000
12,SurGE,surveyforge,factuality_full_text,cit_correct_C,29,0.690517,0.369468,0.000000,0.500000,0.800000,1.000000,1.000000
3,SurGE,perplexity_dr,factuality_full_text,cit_correct_D,39,0.389894,0.105919,0.183099,0.320936,0.383929,0.467385,0.575000


metric,cit_correct_overall,cit_correct_A,cit_correct_B,cit_correct_C,cit_correct_D
model,,,,,
perplexity_dr,0.397437,0.413237,0.322260,0.466654,0.389894
reference,0.592391,0.611337,0.606276,0.507057,0.470113
surveyforge,0.330279,0.352330,0.291343,0.690517,0.296384


## Combined expert + factuality profile

Expert axes are normalized by the maximum value among visible models; factuality axes use the natural `[0, 1]` scale. This mirrors the aggregated radar from the viewer.

In [14]:
def radar_axis_maxima(series: list[tuple[str, list[float | None], str]], width: int) -> list[float]:
    maxima = []
    for idx in range(width):
        vals = [float(values[idx]) for _, values, _ in series if idx < len(values) and values[idx] is not None and not pd.isna(values[idx])]
        max_value = max(vals) if vals else 1.0
        maxima.append(max_value if max_value > 0 else 1.0)
    return maxima


expert_means = mean_values(expert_df, EXPERT_RADAR_METRICS)
factuality_means = mean_values(factuality_df, FACTUALITY_RADAR_METRICS)
overlap_models = sorted(set(expert_means.index).intersection(set(factuality_means.index)), key=str.lower)

combined_rows = []
combined_series = []
for i, model in enumerate(overlap_models):
    expert_values = [expert_means.loc[model, key] if key in expert_means.columns else np.nan for key, _ in EXPERT_RADAR_METRICS]
    factuality_values = [factuality_means.loc[model, key] if key in factuality_means.columns else np.nan for key, _ in FACTUALITY_RADAR_METRICS]
    values = [None if pd.isna(v) else float(v) for v in expert_values + factuality_values]
    combined_series.append((model, values, RADAR_COLORS[i % len(RADAR_COLORS)]))
    combined_rows.append({"model": model, **{key: value for (key, _), value in zip(EXPERT_RADAR_METRICS + FACTUALITY_RADAR_METRICS, values)}})

combined_table = pd.DataFrame(combined_rows).set_index("model") if combined_rows else pd.DataFrame()
display(combined_table)

if combined_series:
    labels = [f"Expert - {label}" for _, label in EXPERT_RADAR_METRICS] + [f"CitCorrect - {label}" for _, label in FACTUALITY_RADAR_METRICS]
    axis_maxima = radar_axis_maxima(combined_series, len(EXPERT_RADAR_METRICS)) + [1.0] * len(FACTUALITY_RADAR_METRICS)
    radar_chart("Average across surveys", labels, combined_series, height=540, axis_maxima=axis_maxima, show_axis_maxima=True).show()
else:
    print("No overlapping expert and factuality metrics by model.")


,m_crit,m_comp_total,m_open,m_mod,cit_correct_overall,cit_correct_A,cit_correct_B,cit_correct_C,cit_correct_D
model,,,,,,,,,
perplexity_dr,0.179740,0.120853,0.009150,0.467553,0.401597,0.421390,0.316765,0.434829,0.377585
reference,0.092778,0.052339,0.004203,0.465806,0.592642,0.649539,0.568119,0.388558,0.501881
surveyforge,0.228415,0.094727,0.005522,0.609634,0.346666,0.377721,0.291571,0.700833,0.296111


## Pairwise deltas

For a pair of models, compute paired deltas over common `(survey_id, metric)` observations. This is the numerical basis for later confidence intervals and paired statistical tests.

In [15]:
def pick_pair(df: pd.DataFrame) -> tuple[str, str] | None:
    models = sorted(df["model"].dropna().unique(), key=str.lower) if not df.empty else []
    if PAIRWISE_MODELS:
        a, b = PAIRWISE_MODELS
        if a in models and b in models and a != b:
            return a, b
    if len(models) >= 2:
        return models[0], models[1]
    return None


def pairwise_delta(df: pd.DataFrame, kind: str, model_a: str, model_b: str) -> pd.DataFrame:
    sub = df[df["kind"] == kind]
    left = sub[sub["model"] == model_a][["survey_id", "metric", "value"]].rename(columns={"value": "a"})
    right = sub[sub["model"] == model_b][["survey_id", "metric", "value"]].rename(columns={"value": "b"})
    out = left.merge(right, on=["survey_id", "metric"], how="inner")
    out["delta_b_minus_a"] = out["b"] - out["a"]
    out["model_a"] = model_a
    out["model_b"] = model_b
    out["kind"] = kind
    return out


def delta_summary(delta_df: pd.DataFrame) -> pd.DataFrame:
    if delta_df.empty:
        return pd.DataFrame()
    rows = []
    for metric, sub in delta_df.groupby("metric"):
        d = sub["delta_b_minus_a"]
        rows.append({
            "metric": metric,
            "n": len(d),
            "mean_delta": d.mean(),
            "std": d.std(ddof=0),
            "min": d.min(),
            "q10": d.quantile(0.10),
            "q25": d.quantile(0.25),
            "median": d.median(),
            "q75": d.quantile(0.75),
            "q90": d.quantile(0.90),
            "max": d.max(),
            "B>A_pct": 100.0 * (d > 0).mean(),
        })
    return pd.DataFrame(rows).set_index("metric")


pair = pick_pair(metric_long)
if pair is None:
    print("Need at least two models for pairwise deltas.")
else:
    model_a, model_b = pair
    print(f"Pair: {model_b} - {model_a}")
    for kind in KINDS:
        delta_df = pairwise_delta(metric_long, kind, model_a, model_b)
        if delta_df.empty:
            print(f"No paired deltas for {kind}.")
            continue
        print(f"\n{kind}")
        display(delta_summary(delta_df))
        for metric, label in specs_available(kind, delta_df.rename(columns={"delta_b_minus_a": "value"})):
            sub = delta_df[delta_df["metric"] == metric]
            if sub.empty:
                continue
            fig = go.Figure()
            fig.add_trace(go.Box(
                y=sub["delta_b_minus_a"],
                name=label,
                boxpoints="all",
                jitter=0.35,
                pointpos=0,
                marker=dict(size=5, opacity=0.65),
                line=dict(width=1),
                customdata=sub["survey_id"],
                hovertemplate="delta: %{y:.4f}<br>survey: %{customdata}<extra></extra>",
            ))
            fig.add_hline(y=0, line_dash="dash", line_color="gray")
            fig.update_layout(title=f"{label}: {model_b} - {model_a}", yaxis_title="delta", showlegend=False, height=360, margin=dict(l=40, r=20, t=45, b=35), xaxis=dict(showticklabels=False))
            fig.show()


Pair: reference - perplexity_dr

factuality


,n,mean_delta,std,min,q10,q25,median,q75,q90,max,B>A_pct
metric,,,,,,,,,,,
cit_correct_A,36,0.226125,0.160052,-0.1310,-0.01400,0.142100,0.22710,0.348275,0.41975,0.4827,88.888889
cit_correct_B,36,0.250183,0.193456,-0.1414,-0.02110,0.143500,0.26280,0.368475,0.50055,0.7162,88.888889
cit_correct_C,32,-0.079209,0.349615,-1.0000,-0.46528,-0.244050,-0.04690,0.107000,0.28235,0.6667,40.625000
cit_correct_D,36,0.125764,0.173906,-0.2301,-0.09710,-0.023400,0.12060,0.281275,0.34975,0.4888,72.222222
cit_correct_overall,36,0.190178,0.152966,-0.2133,-0.00660,0.094175,0.22105,0.302825,0.34580,0.4263,88.888889
n_claims,36,520.666667,597.084072,-309.0000,-188.50000,-36.500000,432.50000,975.500000,1333.50000,1648.0000,69.444444
n_supported,36,411.638889,361.249310,-138.0000,14.00000,111.000000,365.50000,645.750000,921.50000,1406.0000,91.666667



expert


,n,mean_delta,std,min,q10,q25,median,q75,q90,max,B>A_pct
metric,,,,,,,,,,,
m_comp_total,36,-0.066392,0.032398,-0.1515,-0.10950,-0.086600,-0.06210,-0.038225,-0.02885,-0.0138,0.000000
m_crit,36,-0.090967,0.059027,-0.2812,-0.17505,-0.129000,-0.07980,-0.050025,-0.02625,0.0025,2.777778
m_mod,36,-0.004253,0.106394,-0.1841,-0.12535,-0.089325,-0.01440,0.091500,0.12010,0.2333,38.888889
m_open,36,-0.005347,0.006904,-0.0206,-0.01430,-0.008150,-0.00525,-0.001375,0.00085,0.0123,16.666667
n_claims,36,518.138889,595.689011,-309.0000,-187.00000,-37.250000,432.50000,944.250000,1333.50000,1639.0000,69.444444



surge


,n,mean_delta,std,min,q10,q25,median,q75,q90,max,B>A_pct
metric,,,,,,,,,,,
bleu,36,72.101741,10.975082,26.816972,64.912285,70.354487,75.100081,77.761130,80.263760,85.734677,100.000000
citation_count,36,128.555556,97.207523,-1.000000,12.500000,51.750000,108.500000,185.250000,252.500000,417.000000,97.222222
corpus_match_rate,36,-0.059703,0.290086,-0.678600,-0.357100,-0.225125,-0.114150,0.119075,0.321850,0.695200,33.333333
coverage,36,0.944969,0.057204,0.800000,0.857900,0.928100,0.949550,0.987475,1.000000,1.019200,100.000000
logic,36,-0.662464,0.469235,-1.700000,-1.291667,-0.915775,-0.721429,-0.400000,-0.086842,0.450000,11.111111
reference_self_cited,36,-0.750000,0.433013,-1.000000,-1.000000,-1.000000,-1.000000,-0.750000,0.000000,0.000000,0.000000
relevance_paper,36,0.132217,0.210924,-0.250000,-0.116929,-0.016999,0.110819,0.254336,0.382639,0.731209,66.666667
relevance_section,36,-0.039318,0.157622,-0.358987,-0.174675,-0.127982,-0.067943,0.042591,0.190317,0.401587,36.111111
relevance_sentence,36,-0.040646,0.161458,-0.370845,-0.179661,-0.136915,-0.062920,0.036750,0.191556,0.398500,30.555556


## Single-survey detail

Per-survey tables and the same radar/bar diagnostics that are shown in the viewer's detail and comparison pages.

In [16]:
survey_id = str(SURVEY_ID) if SURVEY_ID is not None else (sorted(survey_wide["survey_id"].unique(), key=lambda x: int(x) if str(x).isdigit() else str(x))[0] if not survey_wide.empty else None)
print(f"survey_id = {survey_id}")

if survey_id is not None:
    detail = survey_wide[survey_wide["survey_id"] == survey_id].sort_values(["kind", "model"])
    display(detail)

    fact_detail = detail[detail["kind"] == "factuality"]
    if not fact_detail.empty:
        rows = []
        series = []
        for i, row in enumerate(fact_detail.itertuples(index=False)):
            values = [getattr(row, key, np.nan) if hasattr(row, key) else np.nan for key, _ in FACTUALITY_RADAR_METRICS]
            values = [None if pd.isna(v) else float(v) for v in values]
            rows.append({"model": row.model, **{key: value for (key, _), value in zip(FACTUALITY_RADAR_METRICS, values)}})
            series.append((row.model, values, RADAR_COLORS[i % len(RADAR_COLORS)]))
        display(pd.DataFrame(rows).set_index("model"))
        radar_chart("CitCorrect profile", [label for _, label in FACTUALITY_RADAR_METRICS], series, height=360).show()

        cat_sub = factuality_categories[factuality_categories["survey_id"] == survey_id]
        if not cat_sub.empty:
            display(cat_sub.pivot_table(index="model", columns="category", values=["n", "n_supported", "support_rate", "share"]))
            for model, sub in cat_sub.groupby("model"):
                fig = go.Figure()
                sub = sub.set_index("category").reindex(["A", "B", "C", "D"])
                supported = sub["n_supported"].fillna(0)
                total = sub["n"].fillna(0)
                fig.add_trace(go.Bar(name="Supported", x=list(sub.index), y=supported, marker_color=[FACTUALITY_CATEGORY_COLORS[c] for c in sub.index], opacity=0.9))
                fig.add_trace(go.Bar(name="Unsupported", x=list(sub.index), y=total - supported, marker_color=[FACTUALITY_CATEGORY_COLORS[c] for c in sub.index], opacity=0.35))
                fig.update_layout(title=f"Claim support by category: {model}", barmode="stack", height=280, margin=dict(l=20, r=20, t=45, b=40), yaxis_title="# claims", legend=dict(orientation="h", y=1.1))
                fig.show()

    expert_detail = detail[detail["kind"] == "expert"]
    if not expert_detail.empty:
        display(expert_detail[[c for c in ["model", "n_claims", "m_crit", "m_comp_total", "m_open", "m_mod"] if c in expert_detail.columns]].set_index("model"))
        mod_sub = expert_modality[expert_modality["survey_id"] == survey_id]
        if not mod_sub.empty:
            display(mod_sub.set_index("model")[["1", "2", "3", "4", "5"]])
            series = []
            for i, (_, row) in enumerate(mod_sub.iterrows()):
                values = [float(row[str(level)]) for level in range(1, 6)]
                series.append((row["model"], values, RADAR_COLORS[i % len(RADAR_COLORS)]))
            radar_chart("Modality category mix", [MODALITY_LABELS[str(level)] for level in range(1, 6)], series, height=340).show()

    surge_detail = detail[detail["kind"] == "surge"]
    if not surge_detail.empty:
        surge_cols = ["model"] + [key for key, _ in SURGE_SCALAR_METRICS if key in surge_detail.columns]
        display(surge_detail[surge_cols].set_index("model"))
else:
    print("No survey rows loaded.")


survey_id = 0


,run_name,dataset,raw_model,model,model_key,kind,suffix,survey_id,query,m_crit,m_comp_total,m_open,m_mod,n_claims,cit_correct_overall,cit_correct_A,cit_correct_B,cit_correct_C,cit_correct_D,n_supported,rouge_1,rouge_2,rouge_l,bleu,sh_recall,relevance_paper,relevance_section,relevance_sentence,structure_quality,logic,reference_self_cited,coverage,citation_count,corpus_match_rate
0,SurGE_perplexity_dr_expert_gemma-4-31b_run1,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,expert,_gemma-4-31b_run1,0,A Survey on Edge Computing Systems and Tools,0.1175,0.0989,0.0014,0.4890,698.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
120,SurGE_reference_expert_gemma-4-31b_run1,SurGE,reference,reference,reference,expert,_gemma-4-31b_run1,0,A Survey on Edge Computing Systems and Tools,0.0383,0.0439,0.0000,0.4856,1617.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
228,SurGE_SurveyForge_expert_gemma-4-31b_run1,SurGE,SurveyForge,surveyforge,surveyforge,expert,_gemma-4-31b_run1,0,A Survey on Edge Computing Systems and Tools,0.1715,0.0759,0.0035,0.4811,869.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40,SurGE_perplexity_dr_factuality_gemma-4-31b_run...,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,factuality,_gemma-4-31b_run1_section_abstract_per_ref,0,A Survey on Edge Computing Systems and Tools,NaN,NaN,NaN,NaN,698.0,0.3610,0.3748,0.0370,0.2857,0.4286,252.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
156,SurGE_reference_factuality_gemma-4-31b_run1_se...,SurGE,reference,reference,reference,factuality,_gemma-4-31b_run1_section_full_text_or_abstrac...,0,A Survey on Edge Computing Systems and Tools,NaN,NaN,NaN,NaN,1604.0,0.2294,0.2764,0.1426,0.1515,0.2179,368.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
269,SurGE_surveyforge_factuality_gemma-4-31b_run1_...,SurGE,surveyforge,surveyforge,surveyforge,factuality,_gemma-4-31b_run1_section_abstract_per_ref,0,A Survey on Edge Computing Systems and Tools,NaN,NaN,NaN,NaN,869.0,0.3855,0.3868,0.4138,0.6667,0.3289,335.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,SurGE_perplexity_dr_surge_gemma-4-31b_run1,SurGE,perplexity_dr,perplexity_dr,perplexity_dr,surge,_gemma-4-31b_run1,0,A Survey on Edge Computing Systems and Tools,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.338703,0.055083,0.138291,7.714738,0.772920,0.250000,0.327103,0.324766,1.0,4.642857,1.0,0.0,50.0,0.5400
192,SurGE_reference_surge_gemma-4-31b_run1,SurGE,reference,reference,reference,surge,_gemma-4-31b_run1,0,A Survey on Edge Computing Systems and Tools,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.949337,0.937071,0.946980,93.449415,0.993903,0.114943,0.041096,0.041096,4.0,3.900000,0.0,1.0,87.0,0.1149
307,SurGE_surveyforge_surge_gemma-4-31b_run1,SurGE,surveyforge,surveyforge,surveyforge,surge,_gemma-4-31b_run1,0,A Survey on Edge Computing Systems and Tools,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.382720,0.060620,0.157330,8.469332,0.907362,0.500000,0.387205,0.442761,2.0,4.600000,0.0,0.0,109.0,1.0000


,cit_correct_overall,cit_correct_A,cit_correct_B,cit_correct_C,cit_correct_D
model,,,,,
perplexity_dr,0.3610,0.3748,0.0370,0.2857,0.4286
reference,0.2294,0.2764,0.1426,0.1515,0.2179
surveyforge,0.3855,0.3868,0.4138,0.6667,0.3289


n                    n_supported                       share                               support_rate                              
category           A      B     C     D           A     B     C     D         A         B         C         D            A         B         C         D
model                                                                                                                                                   
perplexity_dr  587.0   27.0  35.0  49.0       220.0   1.0  10.0  21.0  0.840974  0.038682  0.050143  0.070201     0.374787  0.037037  0.285714  0.428571
reference      995.0  498.0  33.0  78.0       275.0  71.0   5.0  17.0  0.620324  0.310474  0.020574  0.048628     0.276382  0.142570  0.151515  0.217949
surveyforge    729.0   58.0   6.0  76.0       282.0  24.0   4.0  25.0  0.838895  0.066743  0.006904  0.087457     0.386831  0.413793  0.666667  0.328947

,n_claims,m_crit,m_comp_total,m_open,m_mod
model,,,,,
perplexity_dr,698.0,0.1175,0.0989,0.0014,0.4890
reference,1617.0,0.0383,0.0439,0.0000,0.4856
surveyforge,869.0,0.1715,0.0759,0.0035,0.4811


,1,2,3,4,5
model,,,,,
perplexity_dr,0.855301,0.010029,0.123209,0.011461,0.0
reference,0.847866,0.011132,0.136673,0.004329,0.0
surveyforge,0.848101,0.009206,0.138090,0.004603,0.0


,rouge_1,rouge_2,rouge_l,bleu,sh_recall,relevance_paper,relevance_section,relevance_sentence,structure_quality,logic,citation_count,corpus_match_rate,coverage,reference_self_cited
model,,,,,,,,,,,,,,
perplexity_dr,0.338703,0.055083,0.138291,7.714738,0.772920,0.250000,0.327103,0.324766,1.0,4.642857,50.0,0.5400,0.0,1.0
reference,0.949337,0.937071,0.946980,93.449415,0.993903,0.114943,0.041096,0.041096,4.0,3.900000,87.0,0.1149,1.0,0.0
surveyforge,0.382720,0.060620,0.157330,8.469332,0.907362,0.500000,0.387205,0.442761,2.0,4.600000,109.0,1.0000,0.0,0.0
